In [ ]:
import pandas as pd

# Path to data frame with WSIs
df_path = r"D:\DATA\abmil_exp2.csv"
df_all = pd.read_csv(df_path)
print(df_all.columns)

# Path to output dir
output_path = r"D:\NOTEBOOKS\Christine\all_slides\feature_summary_new.csv"

# Path to zarr
zarr_dir = r"Q:\NOTEBOOKS\Christine\all_slides\zarr"

In [ ]:
from helper_functions import strings2lists

list_str_cols = ['snomed_code', 'M', 'T', 'snomed_text', 'T_text', 'M_text', 'undersoeger_anonymous', 'T_category', 'M_category']

for col in list_str_cols: 
    df_all[col] = df_all[col].apply(strings2lists)

In [ ]:
class_dict = {
    'Normal Tissue': 0, 
    'Morphology Not Applicable / Insufficient Tissue': 1, 
    'Cellular Changes / Abnormal Tissue Structure': 1,
    'Traumatic Lesions': 1, 
    'Congenital Malformations': 1,
    'Pregnancy-Related Tissues/Changes': 1,
    'Obstruction / Fluid Retention / Cysts': 1, 
    'Mechanical Changes / Architectural Distortion': 1,
    'Inflammation': 1, 
    'Fibrosis': 1, 
    'Degeneration / Necrosis / Atrophy': 1, 
    'Material Deposits': 1, 
    'Resection Margin Free': 0, 
    'Resection Margin Uncertain': 1,
    'Resection Margin Not Free': 1, 
    'Proliferative/Pre-neoplastic Changes': 1, 
    'Benign Neoplasm': 1, 
    'Uncertain / Borderline Neoplasm': 1, 
    'In Situ Neoplasm': 1, 
    'Malignant Neoplasm': 1,
}

df_all["M_idx"] = df_all["M_category"].apply(
    lambda lst: [class_dict[x] for x in lst]
)

# 0: NORMAL
# 1: NOT NORMAL

In [ ]:
df_sub = df_all.copy()

# If more than one M_idx, select most severe
df_sub['M_idx'] = df_sub['M_idx'].apply(lambda x: max(x) if isinstance(x, list) else x)

In [ ]:
df_sub['M_idx'].value_counts()

In [ ]:
from abmil import KFoldPipeline

pipeline = KFoldPipeline(
    df=df_sub,
    filename_col='filename',
    label_col='M_idx',
    feature_key='features_uni',
    tile_key="tiles_224",
    zarr_dir=zarr_dir,
)

In [ ]:
results = pipeline.kfold_cross_validation(n_splits=5, n_epochs=100, max_tiles=5000, resume_from_checkpoints=True, checkpoint_dir="checkpoints/uni")

In [ ]:
pipeline.print_results()

In [ ]:
# Confusion Matrix
# Attention Heatmap
# plot ROC
# Select Top Tiles
# Zoom to Top Tiles
from abmil_NEW import load_model, load_checkpoint, ZarrSlideDataset, validate_ABMIL
from sklearn.model_selection import StratifiedKFold

model, config, label_mapping = load_checkpoint("D:/NOTEBOOKS/Christine/checkpoints/exp2_h-optimus-0/fold_3_auc_0.9601.pt")
print("Num classes (model):", config["n_classes"])
print(label_mapping)

df_sub["label"] = df_sub["M_idx"].map(label_mapping)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_idx, test_idx) in enumerate(skf.split(df_sub, df_sub["label"])):
    if fold_idx == 2:
        test_df = df_sub.iloc[test_idx].reset_index(drop=True)
        break
print(len(test_df))

test_dataset = ZarrSlideDataset(
    df=test_df,
    filename_col="filename",
    label_col="label",
    feature_key=config["feature_key"],
    tile_key=config["tile_key"],
    zarr_dir=zarr_dir,
    max_tiles=config.get("max_tiles", None),
)

good_rows = []

for i, row in df.iterrows():
    slide_path = row["filename"]
    zarr_path = os.path.join(
        zarr_dir, os.path.basename(slide_path).replace(".mrxs", ".zarr")
    )
    try:
        wsi = open_wsi(slide_path, zarr_path)
        _ = wsi.tables[feature_key].X
        good_rows.append(i)
    except:
        print("Removing:", slide_path)

df = df.iloc[good_rows].reset_index(drop=True)

all_labels, all_preds, all_probs = validate_ABMIL(
    model=model,
    val_dataset=test_dataset
)
print("Unique labels in data:", sorted(set(all_labels)))
print("Prob shape:", all_probs.shape)

In [ ]:
from abmil import confusion_matrix_report, auc_score, per_class_auc, plot_roc_curve

confusion_matrix_report(all_labels, all_preds)

auc = auc_score(all_labels, all_probs)
print("Macro AUC:", auc)

per_class_auc(all_labels, all_probs)

plot_roc_curve(all_labels, all_probs)

In [ ]:
import os
import numpy as np
import pandas as pd
from wsidata import open_wsi

feature_keys = {
    "H-optimus-0": "features_h-optimus-0",
    "CONCH": "features_conch",
    "UNI": "features_uni",
}

paths = df_sub["filename"].tolist()
label_map = df_sub.groupby("filename")["M_idx"].max().to_dict()

rows = []

for slide_path in paths:
    print(slide_path)
    zarr_path = os.path.join(zarr_dir, os.path.basename(slide_path).replace(".mrxs", ".zarr"))

    try:
        wsi = open_wsi(slide_path, zarr_path)
    except Exception as e:
        print(e)
        continue

    label = label_map.get(slide_path)

    for model, key in feature_keys.items():
        X = wsi.tables.get(key, {}).X if key in wsi.tables else None
        if X is None or X.size == 0:
            continue

        norms = np.linalg.norm(X, axis=1)
        bag_size = X.shape[0]

        rows.extend([
            {
                "filename": slide_path,
                "model": model,
                "M_idx": label,
                "bag_size": bag_size,
                "feature_norm": float(n),
            }
            for n in norms
        ])

df_plot = pd.DataFrame(rows)

In [ ]:
norm_summary = (
    df_plot
    .groupby("model")["feature_norm"]
    .agg(["count", "mean", "std", "median"])
    .round(4)
)

print("Norm Summary:",
      norm_summary)

bag_df = (df_plot[df_plot["model"] == "H-optimus-0"])

bag_summary = (
    bag_df
    .groupby("M_idx")["bag_size"]
    .agg(["count", "mean", "std", "median"])
    .round(2)
)

print("Bag Summary:",
      bag_summary)

In [ ]:
m_idx_map = {
    0: "0 - healthy (n=315)",
    1: "1 - non-healthy (n=185)",
}
bag_df["M_label"] = bag_df["M_idx"].map(m_isdx_map)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

order = ["0 - healthy (n=315)", "1 - non-healthy (n=185)"]
palette = sns.color_palette(n_colors=len(order))
color_map = dict(zip(order, palette))

sns.set_style("white")

fig, ax = plt.subplots(figsize=(7, 5))

sns.histplot(
    data=bag_df,
    x="bag_size",
    hue="M_label",
    hue_order=order,
    bins=50,
    multiple="stack",
    alpha=0.4,
    palette=color_map,
    ax=ax
)

legend = ax.get_legend()
legend.set_title("M idx")

handles = legend.legend_handles
ymax = ax.get_ylim()[1]

for i, label in enumerate(order):
    values = bag_df.loc[bag_df["M_label"] == label, "bag_size"]

    mean_val = values.mean()
    median_val = values.median()
    color = color_map[label]

    ax.axvline(mean_val, color=color, linestyle="-", linewidth=2)
    ax.axvline(median_val, color=color, linestyle="--", linewidth=2)

    # Mean
    ax.text(
        mean_val,
        ymax * 0.9,
        f"Mean: {mean_val:.1f}",
        color=color,
        ha='left',
        fontsize=8, 
        backgroundcolor='white'
    )
    
    # Median
    ax.text(
        median_val,
        ymax * 0.75,
        f"Median: {median_val:.1f}",
        color=color,
        ha='left',
        fontsize=8,
        fontweight='bold',
        backgroundcolor='white'
    )

ax.set_title("Bag Size Distribution")
ax.set_xlabel("Instances per Slide")
ax.set_ylabel("Count")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure()

sns.set_style("white")

ax = sns.histplot(
    data=df_plot,
    x="feature_norm",
    hue="model",
    bins=100,
    alpha=0.4
)

sns.move_legend(
    ax,
    "upper left",
    bbox_to_anchor=(1, 1),
    title="Model"
)

plt.title("Feature Norm Distribution per Model")
plt.xlabel("L2 Norm")

plt.tight_layout()
plt.show()

In [ ]:
df_plot["norm_zscore"] = df_plot.groupby(["filename", "model"])["feature_norm"] \
    .transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))

In [ ]:
plt.figure()
sns.histplot(
    data=df_plot,
    x="norm_zscore",
    hue="model",
    bins=100,
    alpha=0.4
)
plt.title("Normalized Feature Norm Distribution")
plt.xlabel("Z-scored Norm")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- (1) Raw norms ---
sns.histplot(
    data=df_plot,
    x="feature_norm",
    hue="model",
    bins=100,
    alpha=0.4,
    ax=axes[0],
    legend= True
)
legend = ax.get_legend()
legend.set_title("M idx")
axes[0].set_title("Feature Norm Distribution")
axes[0].set_xlabel("L2 Norm")

# --- (2) Normalized norms ---
sns.histplot(
    data=df_plot,
    x="norm_zscore",
    hue="model",
    bins=100,
    alpha=0.4,
    ax=axes[1],
)
axes[1].set_title("Normalized Norms (Per-Slide)")
axes[1].set_xlabel("Z-scored Norm")

plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import wilcoxon

# per-fold AUC and ACC values for H-optimus-0, CONCH, and UNI
h_AUC = ['0.9249', '0.9352', '0.9601', '0.8910', '0.8983']
h_ACC = ['0.8800', '0.9000', '0.9000', '0.8500', '0.8200']
c_AUC = ['0.8507', '0.8820', '0.9069', '0.8953', '0.8662']
c_ACC = ['0.8400', '0.7900', '0.8400', '0.8500', '0.8000']
u_AUC = ['0.9189', '0.9541', '0.9206', '0.9163', '0.9172']
u_ACC = ['0.8800', '0.8700', '0.8200', '0.8500', '0.8200']

# AUC comparisons
print("AUC:")
print("H-optimus-0 vs CONCH:", wilcoxon(h_AUC, c_AUC))
print("H-optimus-0 vs UNI:", wilcoxon(h_AUC, u_AUC))
print("CONCH vs UNI:", wilcoxon(c_AUC, u_AUC))

# ACC comparisons
print("\nACC:")
print("H-optimus-0 vs CONCH:", wilcoxon(h_ACC, c_ACC))
print("H-optimus-0 vs UNI:", wilcoxon(h_ACC, u_ACC))
print("CONCH vs UNI:", wilcoxon(c_ACC, u_ACC))


In [ ]:
# Multiple testing correction (Bonferroni)
print("\nBonferroni-corrected p-values:")
print("H-optimus-0 vs CONCH (AUC):", wilcoxon(h_AUC, c_AUC).pvalue * 3)
print("H-optimus-0 vs UNI (AUC):", wilcoxon(h_AUC, u_AUC).pvalue * 3)
print("CONCH vs UNI (AUC):", wilcoxon(c_AUC, u_AUC).pvalue * 3)
print("H-optimus-0 vs CONCH (ACC):", wilcoxon(h_ACC, c_ACC).pvalue * 3)
print("H-optimus-0 vs UNI (ACC):", wilcoxon(h_ACC, u_ACC).pvalue * 3)
print("CONCH vs UNI (ACC):", wilcoxon(c_ACC, u_ACC).pvalue * 3)